# ISBI paper: GPU experiments (27 Oct 2026)

The experiments from the 24 Oct strengthening plan that need the GPU. The CPU re-analyses are already done
(`isbi_cpu_reanalyses_27oct/`, report in `out/12_cpu_reanalyses_27oct/REPORT.md`).

| Part | Question | GPU time (RTX 5080, estimate) | Needs |
|---|---|---|---|
| **E1** | Is the IDH signal in the tumour or in the rest of the brain? (input occlusion) | 30-60 min | existing checkpoints + `novelty_input_cache` |
| **E2** | Does the hybrid's specificity hold on an independent, pre-operative IDH-wildtype cohort? (OpenNeuro ds007045) | ~1 h + download (~7 GB) | internet once |
| **E4** | Leave-UCSF-out with the **full three-member** ensemble (CA44 and MC42 retrained without UCSF) | ~5-6 h (two members) | training |
| **E5** | Leave-UTSW-out (second held-out site; UTSW has the most mutants and grade 2-3 tumours) | ~3 h (CA42); ~9 h for all three | training |

**How to run.** Put this notebook in the RadGenome project folder (next to `research_experiments.py`). Set the switches in
Section 1 and *Run All*. Every long step saves to disk and resumes where it stopped, so you can interrupt the kernel at
any time and re-run. After a CUDA error, restart the kernel first. Suggested order: E1 and E2 now (inference only, about
2 h together), then leave E4 and E5 training overnight.

**Nothing here tunes anything on test labels.** E1 uses each patient's held-out checkpoints; E2 uses the frozen models
and frozen thresholds; E4/E5 select checkpoints on inner validation folds of the training sites only.

**Pre-declared outputs and reading rules** are written at the top of each part, before any result appears.

## 1. Configuration

In [ ]:
from pathlib import Path
import os, sys
PROJECT = Path.cwd()                                                        # the RadGenome project folder
PKG = Path.home() / 'Downloads' / 'RadGenome_ISBI_package_v6'               # unzipped v6 package (results/, code/)
CPU_DIR = PROJECT / 'isbi_cpu_reanalyses_27oct'                             # common.py rebuilds the paper's 1,393-patient frame
DL_OUT = Path.home() / 'Downloads' / 'outputs'
BASE = DL_OUT / 'clinical_experiments'                                      # member runs (CA42, CA44, MC42) and LOSO runs
OUT = PROJECT / 'outputs' / 'isbi_gpu_27oct'
INPUT_CACHE = DL_OUT / 'novelty_input_cache'                                # preprocessed crops from the probe run (reused)
PATH_REMAP = {r'C:\Users\claks\Downloads\PKG - UCSF-PDGM Version 5\UCSF-PDGM-v5': r'C:\Users\claks\Downloads\UCSF-PDGM'}
DEVICE = 'cuda'

RUN_E1 = True
RUN_E2 = True
E2_DOWNLOAD = True             # download the ~7 GB of ds007045 files E2 needs (skips files already present and verified)
DS_ROOT = Path.home() / 'Downloads' / 'ds007045'                            # keep it out of OneDrive
DS_FILES = PROJECT / 'ds007045_required_files.csv'                          # paths, sizes and SHA-256 (dataset v2.0.1)
RUN_E4 = True                  # ~5-6 h: CA44 and MC42 without UCSF
RUN_E5 = True                  # ~3 h: CA42 without UTSW
E5_ALL_MEMBERS = False         # True: also CA44 and MC42 without UTSW (another ~6 h)

N_BOOT, SEED = 2000, 20261021
OCCLUSION_CHUNK = 4            # conditions per forward pass; lower to 2 if you hit CUDA out-of-memory

## 2. Setup

In [ ]:
import json, time, warnings, gc, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
warnings.filterwarnings('ignore')
for p in (PROJECT, CPU_DIR):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
os.environ['RG_PKG'] = str(PKG); os.environ['RG_DL_OUT'] = str(DL_OUT); os.environ['RG_RG_OUT'] = str(PROJECT / 'outputs')
import isbi_gpu_experiments as X
from common import build
OUT.mkdir(parents=True, exist_ok=True)
G = build()                    # the paper's analysis frame and helpers, executed from the package's own notebook code
Z, MODELS, run_cv, fit = G['Z'], G['MODELS'], G['run_cv'], G['fit']
cv = run_cv(Z, MODELS)
from sklearn.metrics import roc_auc_score
print(f'Analysis cohort: {len(Z)} patients, {int(Z.y.sum())} IDH-mutant | hybrid AUROC {roc_auc_score(Z.y, cv["Hybrid"]["p"]):.3f} (paper 0.961)')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none - E1/E2 will be very slow, E4/E5 should not be run')
def md_table(df, nd=3):
    f = lambda v: f'{v:.{nd}f}' if isinstance(v, (float, np.floating)) else str(v)
    return '\n'.join(['| ' + ' | '.join(map(str, df.columns)) + ' |', '|' + '---|' * len(df.columns)] +
                     ['| ' + ' | '.join(f(v) for v in r) + ' |' for r in df.itertuples(index=False)])
REPORT = {}

## E1. Region occlusion: is the IDH signal in the tumour or in the brain?

**Why.** Most of the network's positive attribution falls in brain more than 10 mm from the lesion, roughly in proportion
to its volume. A reviewer will ask whether the network reads age, atrophy or scanner from normal brain. Attribution maps
cannot answer that; changing the input can.

**Design (pre-declared).** Every development patient is scored by the three members' **held-out** checkpoints (the same
checkpoints that produced the paper's out-of-fold predictions) under eight input conditions:

| Condition | Input |
|---|---|
| original | unchanged (must reproduce the saved OOF probabilities, tolerance 0.01) |
| tumour_ring_only | lesion + 10 mm ring kept; everything else set to 0, the background value of the normalised input |
| lesion_only | lesion kept; everything else 0 |
| brain_only | lesion + ring set to 0; the rest of the brain kept |
| t1c_blank / flair_blank | one channel set to 0 |
| tumour_ring_only_blurred_brain | lesion + ring kept; the rest of the brain replaced by its Gaussian blur (sigma 4 voxels) |
| brain_only_blurred_tumour | the rest of the brain kept; lesion + ring replaced by their Gaussian blur |

The ensemble probability is the mean of the three members, as in the paper.

**Pre-declared outputs:** AUROC in the 1,393 analysis patients; AUROC in the published age-matched sample (96 mutant /
262 wildtype, same code and seed); within-class Spearman rho of the ensemble logit with age; paired AUROC differences
from the original input; image-only and hybrid models refitted with each occluded ensemble.

**Reading rule (fixed now).** If *tumour_ring_only* keeps most of the original AUROC in the age-matched sample and
*brain_only* falls towards 0.5 there, the signal is tumour-based. If *brain_only* stays high in the age-matched sample,
the network uses information outside the tumour that is not age; report it as a shortcut finding.

**Caveat for the paper.** Occluded inputs are out of distribution, so this is a sensitivity analysis. The two blurred
conditions are milder perturbations of the same contrast and show whether conclusions depend on the fill.

In [ ]:
E1_DIR = OUT / 'e1_occlusion'
if RUN_E1:
    from ensemble_oof_xrai import cohort
    cases = cohort(runs=[BASE / m for m in X.MEMBERS], threshold=X.THRESHOLD)
    cases = [c for c in cases if c['label'] is not None]
    print(f'{len(cases)} development patients with IDH labels | members: {[X.SHORT[m] for m in X.MEMBERS]}')
    t0 = time.time()
    X.run_occlusion(cases, E1_DIR, INPUT_CACHE, device=DEVICE, path_remap=PATH_REMAP, chunk=OCCLUSION_CHUNK)
    print(f'E1 inference finished in {(time.time() - t0) / 60:.1f} min')

In [ ]:
if RUN_E1:
    long, ens = X.load_occlusion(E1_DIR)
    dev = (long.original - long.saved_oof).abs()
    print(f'Original input vs saved OOF probability, per member and patient: max |diff| {dev.max():.4f}; '
          f'{int((dev > 0.01).sum())} of {len(dev)} above 0.01 (AMP vs FP32 rounding is expected to stay well below this)')
    E1a, E1d, E1r, E1info = X.analyse_occlusion(ens, Z, run_cv, MODELS, n_boot=N_BOOT, seed=SEED)
    print(E1info)
    show = ['condition', 'n', 'mutant', 'auroc', 'auroc_ci', 'auroc_age_matched', 'auroc_age_matched_ci',
            'rho_age_wildtype', 'rho_age_wildtype_ci', 'rho_age_mutant', 'rho_age_mutant_ci']
    display(E1a[show].round(3)); display(E1d.round(3)); display(E1r.round(3))
    per_member = long.merge(Z[['patient_id', 'y']], on='patient_id')
    E1m = per_member.groupby('member').apply(lambda g: pd.Series({c: roc_auc_score(g.y, g[c]) for c in X.CONDITIONS})).round(3)
    display(E1m)
    for name, df in [('E1_auroc_by_condition', E1a), ('E1_paired_differences', E1d), ('E1_refitted_models', E1r),
                     ('E1_per_member_auroc', E1m.reset_index())]:
        df.to_csv(E1_DIR / f'{name}.csv', index=False)
    REPORT['E1. Occlusion: ensemble AUROC by input condition'] = E1a[show]
    REPORT['E1. Paired AUROC difference from the original input (site x class bootstrap; age-matched: cluster bootstrap)'] = E1d
    REPORT['E1. Models refitted with the occluded ensemble logit (5x5 CV, AUROC)'] = E1r
    REPORT['E1. Per-member AUROC by condition'] = E1m.reset_index()

    fig, ax = plt.subplots(figsize=(8, 3.6)); xs = np.arange(len(E1a))
    ax.bar(xs - .2, E1a.auroc, .4, color='#2a6fdb', label='all patients')
    ax.bar(xs + .2, E1a.auroc_age_matched, .4, color='#9a9a96', label='age-matched')
    ax.axhline(.5, color='k', lw=.6, ls=':'); ax.set_ylim(.4, 1)
    ax.set_xticks(xs); ax.set_xticklabels(E1a.condition, rotation=30, ha='right', fontsize=8); ax.set_ylabel('ensemble AUROC')
    ax.legend(frameon=False, fontsize=8); ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout(); fig.savefig(E1_DIR / 'E1_occlusion.png', dpi=200); plt.show()

## E2. Pre-operative external specificity: OpenNeuro ds007045

**Data.** Filimonova et al., *Sci Data* 13:1213 (2026), doi:10.1038/s41597-026-07953-2, CC0. 363 adults with primary
glioblastoma from eight European centres, pre-operative T1, T1c, T2, FLAIR; skull-stripped, MNI152-registered,
N4-corrected derivatives; neuroradiologist-reviewed BraTS-label masks; age, MGMT and field strength in `participants.tsv`.
E2 downloads only the N4 T1c, FLAIR and T2 and the masks (~7 GB), checking each file's size and SHA-256.

**Check before you report anything.** The dataset files do not record IDH status. The cohort is described as
"histologically confirmed primary glioblastoma" collected 2018-2025, so some cases may have been diagnosed under WHO 2016,
when "glioblastoma" could include IDH-mutant tumours. Read the paper's inclusion criteria: if all patients are
IDH-wildtype (WHO CNS5, or IDH tested), report specificity; if IDH was not tested in everyone, report the results as
"false-positive rate in a cohort expected to be IDH-wildtype" and say so. Either way there is no AUROC (no mutants).

**Pre-declared outputs.** For every frozen model (frozen coefficients and thresholds; imaging features z-scored within
the cohort, as for MU and RHUH): specificity at the Youden and 90%-sensitivity thresholds with exact binomial CIs; mean
predicted probability of IDH mutation (calibration-in-the-large against an observed rate of 0); false-positive rate by
age band, field strength and centre; compared with development and reserved grade-4 IDH-wildtype specificity.
Sensitivity analysis: features z-scored within centre (centres with >= 20 patients).

**Shift to keep in mind.** Different preprocessing (MNI152, N4) and segmentation (SegResNet, reviewed) from development.
The network crops around the mask, so registration space itself should not matter, but intensity and mask style may.

In [ ]:
E2_DIR = OUT / 'e2_ds007045'; E2_DIR.mkdir(parents=True, exist_ok=True)
if RUN_E2 and E2_DOWNLOAD:
    t0 = time.time()
    status = X.ds_download(DS_ROOT, DS_FILES, workers=8)
    status.to_csv(E2_DIR / 'download_status.csv', index=False)
    failed = status[status.status.str.startswith('FAILED')]
    print(f'{len(status) - len(failed)} of {len(status)} files ready in {(time.time() - t0) / 60:.1f} min')
    if len(failed):
        print('Failed downloads (re-run this cell to retry; or use  pip install openneuro-py  and '
              'openneuro.download(dataset="ds007045", target_dir=DS_ROOT, include=[...]) ):')
        display(failed.head(10))

In [ ]:
if RUN_E2:
    ds_rows, ds_missing = X.ds_rows(DS_ROOT)
    print(f'{len(ds_rows)} subjects with T1c, FLAIR, T2 and mask; {len(ds_missing)} incomplete')
    audit = X.ds_audit(ds_rows, E2_DIR / 'image_audit.csv')
    display(audit.groupby(['shape', 'zooms', 'same_grid']).size().rename('subjects').reset_index())
    display(audit.labels.value_counts().rename('subjects').to_frame())
    bad = audit[(audit.error != '') | ~audit.same_grid.astype(bool) | (audit.lesion_ml <= 0)]
    if len(bad):
        print(f'Excluded ({len(bad)}): grid mismatch, empty mask or unreadable'); display(bad.head(10))
    ds_rows = [r for r in ds_rows if r['patient_id'] not in set(bad.patient_id)]
    print(f'Labels: "1 2 3" = BraTS 2023 (3 = enhancing), handled automatically by the feature code; any "4" = BraTS 2021.')
    # Reproduction check: this machine's inference must reproduce saved OOF probabilities before external scoring
    repro = {X.SHORT[m]: X.reproduce_oof(m, DEVICE, n=3, base=BASE, path_remap=PATH_REMAP) for m in X.MEMBERS}
    print('max |p - saved OOF| on 3 fold-1 validation patients per member:', {k: round(v, 4) for k, v in repro.items()})
    assert max(repro.values()) < 0.01, 'Inference does not reproduce the saved OOF probabilities: stop and check before reading any result'

In [ ]:
if RUN_E2:
    t0 = time.time()
    ds_pred = X.frozen_inference(ds_rows, E2_DIR / 'predictions', device=DEVICE, base=BASE)
    ds_pred.to_csv(E2_DIR / 'ensemble_predictions.csv', index=False)
    print(f'Inference: {(time.time() - t0) / 60:.1f} min | median ensemble P(mutant) {ds_pred.probability.median():.3f}')

Imaging features: the identical extraction code used for development, MU and RHUH (taken from the package's
`build_external_notebook.py`), then the frozen models.

In [ ]:
def external_namespace():
    import nbformat
    nb_path = E2_DIR / '_external_built.ipynb'
    subprocess.run([sys.executable, str(PKG / 'code' / 'build_external_notebook.py'), str(nb_path)], check=True,
                   cwd=str(PKG / 'code'), stdout=subprocess.DEVNULL)
    nb = nbformat.read(nb_path, 4)
    cell = lambda prefix: next(nb.cells[i + 1].source for i, c in enumerate(nb.cells)
                               if c.cell_type == 'markdown' and c.source.lstrip().startswith(prefix))
    cfg = cell('## 1.')
    for old, new in [("FROZEN = PROJECT / 'outputs' / 'idh_hybrid' / 'frozen_models.json'",
                      f"FROZEN = Path(r'{PKG / 'results' / '02_hybrid_model' / 'frozen_models.json'}')"),
                     ("OUTPUT_DIR = PROJECT / 'outputs' / 'idh_hybrid_external'", f"OUTPUT_DIR = Path(r'{E2_DIR}')")]:
        assert cfg.count(old) == 1, old
        cfg = cfg.replace(old, new)
    ns = {'display': display}
    exec(cfg, ns); exec(cell('## 2.'), ns); exec(cell('## 4.'), ns)
    five = cell('## 5.'); exec(five[:five.index('all_rows = []')], ns)
    return ns

if RUN_E2:
    E = external_namespace()
    pred_map = ds_pred.set_index('patient_id').probability
    f = pd.DataFrame([dict(patient_id=r['patient_id'], subject_id=r['subject_id'], scan_id=r['patient_id'], y=0,
                           probability=pred_map[r['patient_id']], age=r['age'], t1c_path=r['t1c_path'], flair_path=r['flair_path'],
                           mask_path=r['tumor_mask_path'], brain_mask_path='', timepoint=0, t2_path=r['t2_path'],
                           adc_path='', fa_path='', perf_path='', adc_kind='', fa_kind='', perf_kind='', site='DS7045',
                           centre=r['centre'], field_strength=r['field_strength'], mgmt_methylated=r['mgmt_methylated']) for r in ds_rows])
    cache = E2_DIR / 'features_DS7045.csv'
    if cache.exists():
        ff = pd.read_csv(cache)
    else:
        from joblib import Parallel, delayed
        from types import SimpleNamespace
        t0 = time.time()
        ff = pd.DataFrame(Parallel(n_jobs=E['N_JOBS'])(delayed(E['extract'])(SimpleNamespace(**r)) for r in f.to_dict('records')))
        ff['scan_id'] = f.scan_id.values; ff.to_csv(cache, index=False)
        print(f'features extracted in {(time.time() - t0) / 60:.1f} min')
    ff['error'] = ff['error'].fillna('').astype(str)
    print('extraction errors:', int((ff.error != '').sum()), ff.loc[ff.error != '', 'error'].value_counts().head(3).to_dict())
    print('label conventions:', ff.label_convention.value_counts().to_dict())
    feat = f.merge(ff.drop(columns=['patient_id', 'site']), on='scan_id')

In [ ]:
def predict_all(d_, frozen):
    out = {}
    for m in ['Age', 'Features', 'Features + age', 'Ensemble (raw)', 'Ensemble (recalibrated)', 'Ensemble + age',
              'Ensemble + features', 'Hybrid']:
        need = [c for c in frozen['models'][m]['inputs'] if c != 'probability']
        ok = d_[need].notna().all(1).values if need else np.ones(len(d_), bool)
        p = np.full(len(d_), np.nan); p[ok] = E['predict'](d_[ok], m); out[m] = p
    cm, cd = np.abs(out['Features + age'] - .5), np.abs(out['Ensemble (recalibrated)'] - .5)
    w = np.where(cm + cd > 0, cm / np.maximum(cm + cd, 1e-12), .5)
    out['Confidence gate (RSNA)'] = w * out['Features + age'] + (1 - w) * out['Ensemble (recalibrated)']
    return out

if RUN_E2:
    frozen = E['frozen']
    TH = {m: dict(youden=frozen['models'][m]['threshold_youden'], sens90=frozen['models'][m]['threshold_sens90']) for m in frozen['models']}
    # gate threshold: Youden point of the gate on development OOF arm probabilities (as in the reserved test)
    dcv = run_cv(Z, {k: MODELS[k] for k in ['Features + age', 'Ensemble (recalibrated)']}, include_raw=False)
    cm, cd = np.abs(dcv['Features + age']['p'] - .5), np.abs(dcv['Ensemble (recalibrated)']['p'] - .5)
    w = np.where(cm + cd > 0, cm / np.maximum(cm + cd, 1e-12), .5); gdev = w * dcv['Features + age']['p'] + (1 - w) * dcv['Ensemble (recalibrated)']['p']
    TH['Confidence gate (RSNA)'] = dict(youden=G['youden_threshold'](Z.y.values, gdev), sens90=G['sens_threshold'](Z.y.values, gdev))
    D2 = E['prepare'](feat)                                   # error rows dropped; features z-scored within the cohort
    D2['age_band'] = pd.cut(D2.age, [0, 40, 55, 200], right=False, labels=['<40', '40-54', '>=55']).astype(str)
    D2['field'] = D2.field_strength.map(lambda v: f'{v:g} T' if np.isfinite(v) else 'unknown')
    D2['centre_group'] = D2.centre.where(D2.centre.map(D2.centre.value_counts()) >= 20, 'other (<20 each)')
    P2 = predict_all(D2, frozen)
    pd.DataFrame(dict(patient_id=D2.patient_id, centre=D2.centre, age=D2.age, field_strength=D2.field_strength,
                      **{m: P2[m] for m in P2})).to_csv(E2_DIR / 'frozen_model_predictions.csv', index=False)
    E2a = X.specificity_table(D2, P2, TH)
    E2b = pd.concat([X.specificity_table(D2, {m: P2[m] for m in ['Features + age', 'Ensemble (raw)', 'Hybrid']}, TH, g)
                     for g in ('age_band', 'field', 'centre_group')])
    E2b = E2b[E2b.operating_point == 'youden']
    # sensitivity analysis: features z-scored within centre (centres with >= 20 patients; the rest pooled)
    raw = feat[feat.error == ''].copy(); raw['centre_group'] = raw.centre.where(raw.centre.map(raw.centre.value_counts()) >= 20, 'other')
    parts = [E['prepare'](g_) for _, g_ in raw.groupby('centre_group')]
    D2c = pd.concat(parts).reset_index(drop=True); P2c = predict_all(D2c, frozen)
    E2c = X.specificity_table(D2c, P2c, TH); E2c = E2c[E2c.operating_point == 'youden']
    # development and reserved comparison: grade-4 IDH-wildtype specificity at the same frozen thresholds
    g4w = (Z.grade_group.values == 'grade 4') & (Z.y.values == 0)
    rp = pd.read_csv(PKG / 'results' / '11_reserved_test' / 'reserved_predictions.csv'); r4w = (rp.grade_group == 'grade 4') & (rp.y == 0)
    comp = []
    for m in ['Age', 'Features', 'Features + age', 'Ensemble (raw)', 'Ensemble + age', 'Ensemble + features', 'Hybrid']:
        th = TH[m]['youden']
        comp.append(dict(model=m, threshold=th,
                         development_grade4_wildtype=f"{(cv[m]['p'][g4w] < th).mean():.3f} (n={int(g4w.sum())}, OOF probabilities)",
                         reserved_grade4_wildtype=f"{(rp.loc[r4w, m] < th).mean():.3f} (n={int(r4w.sum())})",
                         ds007045=f"{(P2[m][np.isfinite(P2[m])] < th).mean():.3f} (n={int(np.isfinite(P2[m]).sum())})"))
    E2d = pd.DataFrame(comp)
    for name, df in [('E2_specificity', E2a), ('E2_specificity_by_group', E2b), ('E2_specificity_within_centre_z', E2c), ('E2_vs_development', E2d)]:
        df.to_csv(E2_DIR / f'{name}.csv', index=False)
    display(E2a.round(3)); display(E2d); display(E2b.round(3)); display(E2c.round(3))
    REPORT['E2. ds007045: specificity at frozen thresholds (exact binomial 95% CI)'] = E2a
    REPORT['E2. Specificity at the Youden threshold: development and reserved grade-4 IDH-wildtype vs ds007045'] = E2d
    REPORT['E2. Specificity by age band, field strength and centre (Youden threshold)'] = E2b
    REPORT['E2. Sensitivity analysis: features z-scored within centre'] = E2c
    fig, ax = plt.subplots(figsize=(6, 3.4))
    bins = np.linspace(0, 1, 41)
    ax.hist(cv['Ensemble (raw)']['p'][g4w], bins, density=True, alpha=.5, color='#9a9a96', label='development grade-4 IDH-wt (OOF)')
    ax.hist(P2['Ensemble (raw)'], bins, density=True, alpha=.5, color='#2a6fdb', label='ds007045')
    ax.axvline(TH['Ensemble (raw)']['youden'], color='k', lw=.8, ls='--'); ax.set_xlabel('ensemble P(IDH-mutant)')
    ax.legend(frameon=False, fontsize=8); ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout(); fig.savefig(E2_DIR / 'E2_ensemble_probabilities.png', dpi=200); plt.show()

## E4. Leave-UCSF-out with the full three-member ensemble

The paper's leave-UCSF-out retrained only CA42 (`clinical_experiments/loso_UCSF`). Here CA44 and MC42 are retrained the
same way: inner 5-fold CV on UTSW + UPENN only, checkpoint selection on inner validation folds, all five inner checkpoints
averaged on UCSF. The inner folds are **identical to loso_UCSF's** (asserted), so the three members' inner out-of-fold
predictions line up and the combiners can be fitted on UTSW + UPENN without any model having seen UCSF.

**Pre-declared outputs (UCSF, n = 433, 68 mutant):** AUROC of the retrained three-member ensemble, features + age,
ensemble + age, ensemble + features and the hybrid, all fitted on UTSW + UPENN; paired differences hybrid minus
features + age and minus ensemble + age (class-stratified bootstrap).

Each member takes about 2.5-3 h. The run resumes from the last saved epoch if interrupted.

In [ ]:
def run_loso(site, members, reference_folds=None):
    dirs = []
    for m in members:
        d_ = BASE / ('loso_UCSF' if (site == 'UCSF' and m == 'v2_reference_seed42') else f'loso_{site}_{X.SHORT[m]}')
        if not (d_ / 'outer_site_predictions.csv').exists():
            X.loso_member(m, site, d_, device=DEVICE, base=BASE, reference_folds=reference_folds, path_remap=PATH_REMAP)
        dirs.append(d_)
        if reference_folds is None: reference_folds = d_ / 'inner_cv'
    return dirs

def report_loso(site, dirs):
    done = [d_ for d_ in dirs if (d_ / 'outer_site_predictions.csv').exists()]
    res, diffs, info = X.loso_combine(site, done, Z, fit, n_boot=N_BOOT, seed=SEED)
    print(f'Leave-{site}-out with {info["members"]} retrained member(s): {[d_.name for d_ in done]} | '
          f'retrained ensemble inner-OOF AUROC on the training sites {info["inner_oof_auroc_training_sites"]:.3f}')
    per = []
    for d_ in done:
        o = pd.read_csv(d_ / 'outer_site_predictions.csv').merge(Z[['patient_id', 'y']], on='patient_id')
        per.append(dict(member_run=d_.name, n=len(o), auroc=roc_auc_score(o.y, o.idh_probability)))
    per = pd.DataFrame(per)
    display(res.round(3)); display(diffs.round(3)); display(per.round(3))
    for name, df in [('results', res), ('paired_differences', diffs), ('per_member', per)]:
        df.to_csv(OUT / f'loso_{site}_{name}.csv', index=False)
    REPORT[f'Leave-{site}-out, {info["members"]} retrained member(s): AUROC on {site}'] = res
    REPORT[f'Leave-{site}-out: paired differences'] = diffs
    REPORT[f'Leave-{site}-out: each retrained member on {site}'] = per

if RUN_E4:
    t0 = time.time()
    dirs_ucsf = run_loso('UCSF', X.MEMBERS, reference_folds=BASE / 'loso_UCSF' / 'inner_cv')
    print(f'E4 training finished in {(time.time() - t0) / 3600:.1f} h')
    report_loso('UCSF', dirs_ucsf)

## E5. Leave-UTSW-out

UTSW holds the most IDH-mutant patients (108 in the analysis cohort) and most grade 2-3 tumours, so it is the most
informative second held-out site (UPENN has only 10 mutants). Same design as E4. By default only CA42 is retrained
(about 3 h); set `E5_ALL_MEMBERS = True` for the full three-member version. The earlier `loso_UTSW` folder (unfinished,
no completed fold) is left untouched; this run writes to `loso_UTSW_CA42` etc.

In [ ]:
if RUN_E5:
    t0 = time.time()
    members = X.MEMBERS if E5_ALL_MEMBERS else ['v2_reference_seed42']
    dirs_utsw = run_loso('UTSW', members)
    print(f'E5 training finished in {(time.time() - t0) / 3600:.1f} h')
    report_loso('UTSW', dirs_utsw)

## Report

In [ ]:
lines = ['# ISBI GPU experiments (27 Oct 2026)', '', f'Run {time.strftime("%Y-%m-%d %H:%M")} on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}.', '']
if RUN_E1: lines += ['E1 info: ' + json.dumps(E1info), '']
for title, df in REPORT.items():
    lines += [f'## {title}', '', md_table(df), '']
lines += ['## Notes', '',
          '- E1 occluded inputs are out of distribution: a sensitivity analysis, not a causal map. The blurred conditions test the fill.',
          '- E2: confirm in Filimonova et al. 2026 that every patient is IDH-wildtype before calling the false-positive rate a specificity.',
          '- E2 features are z-scored within the external cohort (as for MU and RHUH); thresholds and coefficients are frozen.',
          '- E4/E5: architecture and recipe were chosen in cross-validation that included the held-out site; only the weights are site-naive.']
(OUT / 'REPORT.md').write_text('\n'.join(lines), encoding='utf-8'); print('Wrote', OUT / 'REPORT.md')